In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter

# Setup root directory paths
ROOT = Path("D:/Bussiness_plan/Multimodal_PM25")
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thiết lập phong cách hiển thị hình vẽ (Aesthetics)
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 15,
    "legend.fontsize": 10,
    "figure.dpi": 200
})

def simulate_pm25_signal():
    np.random.seed(100)
    hours = np.arange(168) # 1 tuần dữ liệu theo giờ
    
    # 1. Tạo tín hiệu sạch gốc: Chu kỳ ngày đêm (12 ug/m3) + Đỉnh ô nhiễm sương mù kéo dài (85 ug/m3, ngày 3-5)
    daily_cycle = 12 * np.sin(2 * np.pi * hours / 24)
    haze_event = 85 * np.exp(-((hours - 96) / 28)**2) # Đỉnh ô nhiễm lớn nhất rơi vào giờ thứ 96
    baseline = 40.0
    true_signal = baseline + daily_cycle + haze_event
    
    # 2. Thêm nhiễu ngẫu nhiên đo đạc thông thường
    noise = np.random.normal(0, 3.5, len(hours))
    raw_observations = true_signal + noise
    
    # 3. Chèn các điểm dị biệt đột biến (Outliers - Lỗi hiệu chuẩn cảm biến trong 1 giờ)
    outlier_indices = [30, 72, 130]
    outlier_values = [190.0, 225.0, 180.0]
    for idx, val in zip(outlier_indices, outlier_values):
        raw_observations[idx] = val
        
    return hours, true_signal, raw_observations, outlier_indices

def detect_outliers_mad(raw, window=13, threshold=3.5):
    # Sử dụng Rolling Median Absolute Deviation (MAD) để phát hiện dị biệt
    s = pd.Series(raw)
    rolling_median = s.rolling(window, center=True, min_periods=1).median()
    rolling_mad = (s - rolling_median).abs().rolling(window, center=True, min_periods=1).median()
    
    rolling_mad = rolling_mad.replace(0, 1.0) # Tránh lỗi chia cho 0
    modified_z = 0.6745 * (s - rolling_median) / rolling_mad
    outliers_mask = modified_z.abs() > threshold
    
    # Thay thế các điểm dị biệt bằng trung vị trượt để chuẩn bị làm mịn
    filtered = s.copy()
    filtered[outliers_mask] = rolling_median[outliers_mask]
    
    return outliers_mask, filtered.values

def evaluate_filters(true_sig, raw, filtered):
    # Áp dụng các bộ lọc làm mịn khác nhau trên tín hiệu sau khi đã loại bỏ dị biệt
    # 1. Moving Average (Cửa sổ 24 giờ)
    ma_24 = pd.Series(filtered).rolling(24, center=True, min_periods=1).mean().values
    
    # 2. Exponential Moving Average (alpha=0.1)
    ema = pd.Series(filtered).ewm(alpha=0.1, adjust=False).mean().values
    
    # 3. Savitzky-Golay (Cửa sổ 15 giờ, đa thức bậc 2)
    savgol_15 = savgol_filter(filtered, window_length=15, polyorder=2)
    
    # 4. Savitzky-Golay (Cửa sổ 31 giờ, đa thức bậc 2) - Làm mịn quá đà (over-smoothing)
    savgol_31 = savgol_filter(filtered, window_length=31, polyorder=2)
    
    def get_filter_metrics(smooth):
        residual = true_sig - smooth
        # Tỷ số Tín hiệu trên Nhiễu (SNR)
        snr = 10 * np.log10(np.var(true_sig) / np.var(residual))
        
        # Sai số bảo toàn đỉnh ô nhiễm lớn nhất (tại giờ 96)
        true_peak_val = np.max(true_sig)
        peak_idx = np.argmax(true_sig)
        peak_pres_err = true_peak_val - smooth[peak_idx]
        mean_res = np.mean(residual)
        return snr, peak_pres_err, mean_res

    methods = {
        "Moving Average (24h)": (ma_24, "24 Hours"),
        "Exponential Moving Average (alpha=0.1)": (ema, "N/A"),
        "Savitzky-Golay (Window 15, poly 2)": (savgol_15, "15 Hours"),
        "Savitzky-Golay (Window 31, poly 2)": (savgol_31, "31 Hours")
    }
    
    rows = []
    for name, (smooth, win) in methods.items():
        snr, peak_err, mean_res = get_filter_metrics(smooth)
        rows.append({
            "Smoothing Method": name,
            "Window Size": win,
            "SNR (dB)": f"{snr:.2f} dB",
            "Peak Preservation Error (ug/m3)": f"{peak_err:.2f}",
            "Mean Residual": f"{mean_res:.3f}"
        })
        
    return pd.DataFrame(rows), savgol_15

def plot_trajectory(hours, raw, outliers_mask, smoothed, true_sig):
    plt.figure(figsize=(14, 6.5))
    
    # Vẽ chuỗi quan sát thô ban đầu
    plt.plot(hours, raw, color="gray", alpha=0.4, label="Raw PM2.5 Observations", lw=1.2)
    
    # Vẽ đường xu hướng sau làm mịn (Savitzky-Golay)
    plt.plot(hours, smoothed, color="#1B9E77", label="Smoothed Trendline (Savitzky-Golay, w=15)", lw=2.5)
    
    # Vẽ tín hiệu sạch làm tham chiếu
    plt.plot(hours, true_sig, color="black", linestyle=":", label="True Signal (Underlying Trend)", lw=1.5)
    
    # Vẽ các điểm dị biệt bị gắn cờ đỏ
    outlier_idx = np.where(outliers_mask)[0]
    plt.scatter(outlier_idx, raw[outlier_idx], color="red", marker="x", s=80, zorder=5, label="Flagged Outliers (Calibration Errors)")
    
    plt.title("Figure 3: Time-Series Trajectory of Raw, Outlier-Filtered, and Smoothed Observations", pad=15, fontweight="bold")
    plt.xlabel("Time (Hours)")
    plt.ylabel("PM2.5 Concentration (ug/m3)")
    plt.xlim(0, 168)
    plt.ylim(0, 250)
    
    # Chú thích đỉnh ô nhiễm sương mù mùa đông được bảo toàn nguyên vẹn
    plt.annotate(
        "Preserved Winter Haze Hump\n(True Peak = 137 ug/m3)",
        xy=(96, 137),
        xytext=(120, 160),
        arrowprops=dict(facecolor='black', shrink=0.08, width=1, headwidth=6),
        fontsize=9.5,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.8, ec="orange", lw=0.8)
    )
    
    plt.legend(loc="upper right", frameon=True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "smoothing_analysis.png", bbox_inches="tight", dpi=300)
    print(f"Saved Figure 3 to: {OUTPUT_DIR / 'smoothing_analysis.png'}")
    plt.close()

if __name__ == "__main__":
    hours, true_sig, raw, _ = simulate_pm25_signal()
    outliers_mask, filtered = detect_outliers_mad(raw)
    df_table, smoothed = evaluate_filters(true_sig, raw, filtered)
    plot_trajectory(hours, raw, outliers_mask, smoothed, true_sig)
    
    # In Bảng 3 định dạng Markdown
    print("\n=== Table 3: Influence of Smoothing Window and Method on Signal Fidelity ===")
    headers = list(df_table.columns)
    md_table = "| " + " | ".join(headers) + " |\n"
    md_table += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    for _, row in df_table.iterrows():
        md_table += "| " + " | ".join(str(val) for val in row) + " |\n"
    print(md_table)
    
    df_table.to_csv(OUTPUT_DIR / "smoothing_table.csv", index=False)


Saved Figure 3 to: D:\Bussiness_plan\Multimodal_PM25\outputs\smoothing_analysis.png

=== Table 3: Influence of Smoothing Window and Method on Signal Fidelity ===
| Smoothing Method | Window Size | SNR (dB) | Peak Preservation Error (ug/m3) | Mean Residual |
| --- | --- | --- | --- | --- |
| Moving Average (24h) | 24 Hours | 11.09 dB | 17.16 | 0.313 |
| Exponential Moving Average (alpha=0.1) | N/A | 7.26 dB | 16.88 | 0.619 |
| Savitzky-Golay (Window 15, poly 2) | 15 Hours | 27.00 dB | 5.49 | 0.401 |
| Savitzky-Golay (Window 31, poly 2) | 31 Hours | 15.68 dB | 8.51 | 0.418 |



: 